In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, PReLU, ELU, Activation, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

# ------------------------------
# Custom Activation Layers
# ------------------------------

class Swish(Layer):
    def call(self, inputs):
        return inputs * tf.nn.sigmoid(inputs)

class Mish(Layer):
    def call(self, inputs):
        return inputs * tf.math.tanh(tf.math.softplus(inputs))

# ------------------------------
# Load and Trim Trace Data
# ------------------------------

X_full = np.load('2208_A[8]_rand_traces_full_algo_IV(FIXED)_Keys(1Lakh).npy')[:, :2000]

key_files = {
    1: np.load('A8_last_1_bit_array.npy'),
    2: np.load('A8_last_2_bits_array.npy'),
    4: np.load('A8_last_4_bits_array.npy'),
    8: np.load('A8_last_8_bits_array.npy'),
}

# ------------------------------
# Apply PCA
# ------------------------------

pca = PCA(n_components=0.99)
X_full_pca = pca.fit_transform(X_full)

# ------------------------------
# Activation Functions Dictionary
# ------------------------------

activation_layers = {
    'relu': lambda: Activation('relu'),
    'leaky_relu': lambda: LeakyReLU(alpha=0.1),
    'prelu': lambda: PReLU(),
    'elu': lambda: ELU(alpha=1.0),
    'selu': lambda: Activation('selu'),
    'swish': lambda: Swish(),
    'mish': lambda: Mish(),
}

# ------------------------------
# Training Loop
# ------------------------------

results = []

for bits, y_full in key_files.items():
    num_classes = 2 ** bits
    y_full_cat = to_categorical(y_full, num_classes=num_classes)

    X_train, X_test, y_train, y_test = train_test_split(
        X_full_pca, y_full_cat, test_size=0.2, random_state=42
    )

    for act_name, act_layer in activation_layers.items():
        print(f"\n🔄 Training for {bits}-bit classification using {act_name}...")

        model = Sequential([
            Dense(512, input_shape=(X_train.shape[1],)), act_layer(),
            Dense(256), act_layer(),
            Dense(128), act_layer(),
            Dense(num_classes, activation='softmax')
        ])

        model.compile(
            optimizer=Adam(0.0001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=100,
            batch_size=256,
            verbose=0
        )

        y_test_pred = model.predict(X_test, verbose=0)
        test_acc = accuracy_score(
            np.argmax(y_test, axis=1),
            np.argmax(y_test_pred, axis=1)
        )

        results.append({
            'bits': bits,
            'activation': act_name,
            'final_test_accuracy': test_acc,
            'train_loss': history.history['loss'][-1],
            'val_loss': history.history['val_loss'][-1],
            'train_accuracy': history.history['accuracy'][-1],
            'val_accuracy': history.history['val_accuracy'][-1]
        })

# ------------------------------
# Save and Display Results
# ------------------------------

results_df = pd.DataFrame(results)

# Sort and round for readability
results_df = results_df.sort_values(by=['bits', 'activation'])
results_df = results_df.round({
    'final_test_accuracy': 4,
    'train_loss': 4,
    'val_loss': 4,
    'train_accuracy': 4,
    'val_accuracy': 4,
})

# Display table in console
print("\n✅ Activation Function Performance Summary:\n")
print(results_df.to_string(index=False))

# Optionally save to CSV
results_df.to_csv('activation_comparison_all_bits.csv', index=False)
print("\n📁 Results saved to 'activation_comparison_all_bits.csv'")